# Learn-to-Edit — can editability be *induced*?  (frozen learned editor + light fine-tune)

**Direction:** `research/directions/learn-to-edit.md` · `[reframe]` · sub-Q3 (editability).

**Premise.** Hand-crafted edits fail (ghost/revert) because the GRU hidden state `h` is non-canonical: a curved, history-carrying embedding of `(pos,vel)`; **readable ≠ controllable**. The failed baselines — linear-probe pseudo-inverse, global-manifold projection, obs-gradient (per-sample optimization landing far off-manifold, resid ~15.7) — all used the *wrong map* from target → `h_edit`. This notebook asks the constructive question: is the failure *fundamental to the representation*, or a failure of the *editing method*?

**Variant A — frozen-model learned editor (information-presence test).** Freeze the trained GRU. Train a small net `E_θ:(h, target_8d) → Δh` on the **edits split** with an **obs-space loss**: roll the FROZEN dynamics forward from `h_edit = h + Δh` and match the GT post-edit clean-observation sequence over `K` steps (+ small on-manifold penalty). Only `E_θ` trains. **Evaluate on HELD-OUT edits** — the train/eval split is what separates *controllability* from *memorization*.

**Variant B — light fine-tune for editability (inducibility test).** Lightly fine-tune the GRU on the edits split so a *fixed simple editor* (the linear-probe pseudo-inverse) induces the intended rollout; then RE-RUN the fiber-collapse + geometry diagnostics on the fine-tuned model — did inducing editability make the state **more canonical**?

**Interpretation guard.** A frozen editor that only works on trained edits = memorization, not controllability. If it induces clean, persistent, selective edits on *held-out* edits → information is present & reachable. If it cannot even overfit → strong structural claim. We report the train↔held-out gap and data-scaling honestly.

**Metrics (obs-space, where the effect lives):** (i) distance to GT post-edit rollout; (ii) ghost ratio (old object vanishes); (iii) persistence (per-step revert curve); (iv) selectivity (non-edited object stays put); (v) off-manifold residual of `h_edit`. Head-to-head vs the failed baselines. Both rich plots AND printed tables. PNGs → `/tmp/learn_to_edit/`.

Conventions: numbered code cells `# [N]`, numbered figures `Fig K`. Paths 3-deep. Mirrors `canonical_state_editing.ipynb`.

---
## 1 — Setup: frozen GRU, data, teacher-forced state bank, `(pos,vel)` probe, global manifold

In [ ]:
# [1] Imports + config + frozen-checkpoint/data load + teacher-forcing + probe + global manifold.
import sys, os, time
sys.path.insert(0, "../../../..")   # repo root -> import pim
sys.path.insert(0, "../../..")      # notebooks/ -> helpers

from dataclasses import replace
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import display
import h5py

import pim.eval as eval
from pim.extractors import LinearExtractor, StateDefinition
from pim.editors import (
    probe_decomposition, inject_state,
    fit_state_subspace, offmanifold_residual, manifold_steer,
)
from pim.world_models import load_checkpoint, load_dataset, make_test_loader
from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene

torch.manual_seed(0); np.random.seed(0)

CHECKPOINT_PATH = "../../../../runs/gru/3_dset3_gru_persistentids_inview_400epochs/best_model.pt"
DATA_DIR        = "../../../../datasets/4_fixed_refl_inview"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ           = 2
SUBSPACE_VAR    = 0.90
OUT = "/tmp/learn_to_edit"
os.makedirs(OUT, exist_ok=True)

# Frozen world model (load_checkpoint sets eval + requires_grad_(False) on all params).
model, ckpt_info = load_checkpoint(CHECKPOINT_PATH, device=DEVICE)
bundle = load_dataset(DATA_DIR, n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
H  = model.hidden_size
EF = edits.edit_frame
DT = float(test.config["dataset"]["sim"]["dt"])
print(f"Model : {ckpt_info.run_name} (epoch {ckpt_info.epoch}, val_loss={ckpt_info.val_loss:.5f})  H={H}")
print(f"Edits : n={edits.n_samples}  edit_frame={EF}  T={edits.T_frames}  obs_res={edits.obs_res}  dt={DT}")

# Teacher-forced state bank on TEST (for probe + manifold; NOT edit training data).
test_loader = make_test_loader(test, batch_size=512, num_workers=6)
preds_tf, states_tf = eval.teacher_force(model, test_loader, device=DEVICE)   # (N,39,H)

# (pos,vel) aligned targets on test for the linear probe.
v_test = h5py.File(test.h5_path, "r")["velocities"][:, :, :N_OBJ, :].astype(np.float32)
vel_tf = v_test[:, :-1, :, :]
pos_tf = test.positions[:, :-1, :N_OBJ, :]
vis_tf = test.is_visible[:, :-1, :N_OBJ].all(axis=2)
posflat = pos_tf.reshape(*pos_tf.shape[:2], N_OBJ*2)
velflat = vel_tf.reshape(*vel_tf.shape[:2], N_OBJ*2)
posvel_tf = np.concatenate([posflat, velflat], axis=-1)                       # (N,39,8)

# Linear (pos,vel) probe — the map the failed pseudo-inverse editor uses.
COMP = ["pos x0","pos y0","pos x1","pos y1","vel x0","vel y0","vel x1","vel y1"]
pv_sdef = StateDefinition(name="posvel", state_shape=(8,), extract_fn=lambda b: b["x"])
linear_pv = LinearExtractor(H, pv_sdef, use_lstsq=True)
linear_pv.fit(states_tf, posvel_tf, mask=vis_tf, device=DEVICE)
linear_pv = linear_pv.to(DEVICE).eval()
Apv, bpv, Apv_pinv = probe_decomposition(linear_pv)  # A:(8,H) b:(8,) pinv:(H,8)

# Position-only probe (for the position-only pseudo-inverse baseline).
pos_sdef = StateDefinition(name="pos", state_shape=(N_OBJ,2), extract_fn=lambda b: b["positions"])
linear_pos = LinearExtractor(H, pos_sdef, use_lstsq=True)
linear_pos.fit(states_tf, pos_tf, mask=vis_tf, device=DEVICE)
linear_pos = linear_pos.to(DEVICE).eval()
Ap, bp, Ap_pinv = probe_decomposition(linear_pos)

# Global state-manifold subspace (PCA) for off-manifold residual + manifold-steer baseline.
subspace = fit_state_subspace(states_tf, var_threshold=SUBSPACE_VAR)
subspace_dev = replace(subspace,
    mean=subspace.mean.to(DEVICE), basis=subspace.basis.to(DEVICE),
    explained_variance_ratio=subspace.explained_variance_ratio.to(DEVICE))
_bank = states_tf.reshape(-1, H)
# real_res_global = REAL-state off-manifold residual (the ~1.7 reference used throughout).
# Comparison anchors cited inline: obs-gradient editor ~15.7 off-manifold (candidate-editability.md);
# GRU fiber resid 0.337 (diagnostic-corrections.md Sec.2) used in Variant B canonicity.
real_res_global = float(offmanifold_residual(
    torch.from_numpy(_bank[:8000]).float().to(DEVICE), subspace_dev).mean())
print(f"probe fit done. global subspace kept {subspace.n_components}/{H} comps "
      f"({subspace.total_explained:.4f} var). REAL-state off-manifold residual = {real_res_global:.4f}")

---
## Definitions — metrics, formulas, units, and direction

Every metric used below is defined here **once**, with its explicit formula and better-direction (↓ = lower better, ↑ = higher better). Later figures/tables reference these names only; the formula lives here.

Notation: `obs(s)` = model-generated observation (ray-intensity vector, length `R = obs_res`) at rollout step `s∈{0..K-1}`, decoded from the edited state `h_edit` rolled forward by the (frozen) dynamics; `obs_u(s)` = the same rollout from the **unsteered** `h_at_edit` (no edit); `gt(s)` = the **clean** (noise-free) GT post-edit observation at step `s`; `tgt` = the single-frame render of the desired post-edit scene; `ghost` rays = rays the edited object occupied pre-edit but should now vacate; `sel` rays = rays owned by the **non-edited** object in the target render. `RMS(·)` = root-mean-square over the listed indices; `‖·‖` = L2 norm.

| metric | symbol | formula | units | better |
|---|---|---|---|---|
| obs-dist to GT rollout | `d_gt` | `sqrt( mean_{s,batch,ray} (obs(s) − gt(s))² )` — **RMSE** over all K steps | intensity (RMSE) | ↓ |
| obs-dist to GT, per step | `d_gt_step[s]` | `sqrt( mean_{batch,ray} (obs(s) − gt(s))² )` | intensity (RMSE) | ↓ |
| dist to target render, step 0 | `d_tgt(s0)` | `sqrt( mean_{batch,ray} (obs(0) − tgt)² )` | intensity (RMSE) | ↓ |
| ghost ratio | `ghost` | `mean_{ghost rays} obs(0) / mean_{ghost rays} obs_u(0)` — intensity left in the vacated zone, normalized by unsteered | ratio (1 = object still there, 0 = gone) | ↓ |
| ghost ratio, per step | `ghost_step[s]` | `mean_{ghost} obs(s) / mean_{ghost} obs_u(0)` | ratio | ↓ |
| selectivity error | `sel_err` | `sqrt( mean_{sel rays} (obs(0) − gt(0))² )` — did the non-edited object move? | intensity (RMSE) | ↓ |
| selectivity, per step | `sel_step[s]` | `sqrt( mean_{sel} (obs(s) − gt(s))² )` | intensity (RMSE) | ↓ |
| off-manifold residual | `resid` | `mean_batch ‖h_edit − proj_S(h_edit)‖`, `S` = global 90%-var PCA of visited states | latent L2 | ↓ (want ≈ real-state ref) |

**Canonicity metrics** (Variant B only, cell [14]) — do NOT confuse with obs-space error:

| metric | formula | units | "more canonical" |
|---|---|---|---|
| fiber residual (MLP) | `‖h − g(pos,vel)‖ / ‖h‖`, `g` = MLP fit | fraction | ↓ |
| R²(h) from (pos,vel) | `1 − Σ(h−g)² / Σ(h−h̄)²` | fraction | ↑ |
| linear pos / vel R² (readability) | linear-probe R² of pos / vel from `h` | fraction | ↑ |
| PCA comps @90% var | # PCA components for 90% state variance | count | ↓ (lower-dim) |

Reference constants (cited inline where used): **real-state off-manifold residual ≈ 1.7** (this notebook, cell [1]); **obs-gradient editor historically lands ~15.7 off-manifold** (`research/scratch/candidate-editability.md`); **GRU fiber residual ≈ 0.337 (MLP), R²(h) ≈ 0.867** (`research/scratch/2026-07-08-diagnostic-corrections.md`, Sec. 2) — Variant B's canonicity re-measurement compares against these.

**All obs-space error quantities here are RMSE** (intensity units), matching the rest of the repo — training-loss curves included (we sqrt the stored MSE at plot time).

---
## 2 — Edit dataset: warm-up to edit, `(pos,vel)` targets, GT post-edit rollout, TRAIN/HELD-OUT split

Warm-force each edit sample to `edit_frame` → `h_at_edit`. Target = teleported `(pos, preserved_vel)` (8-dim). GT supervision = the **clean** post-edit observation sequence over `K` steps. **Same small data budget as the probes:** default `N_TRAIN=256`. Held-out = disjoint unseen edits. We render the TARGET scene and PRE-edit scene per sample to define the ghost zone (rays the edited object vacates) and the selectivity zone (rays owned by the non-edited object).

In [ ]:
# [2] Warm-up to edit, build 8-dim (pos,vel) targets, GT clean rollout, TRAIN/HELD-OUT split.
N_POOL   = 3000     # pool of edit samples we materialize h_at_edit / targets / GT for
N_TRAIN  = 256      # few-shot budget matching the probes (data-scaling swept later in Section 4)
N_HELD   = 512      # held-out (unseen) edits for the controllability test
K        = 15       # rollout horizon for obs-space loss + metrics  (post-edit frames available = T-EF-1 = 19)
assert EF + K <= edits.T_frames

warm = eval.warm_up_to_edit(model, edits.obs[:N_POOL], EF, n_viz=1, n_ctx_show=1, device=DEVICE)
H0 = torch.from_numpy(warm.h_at_edit[:N_POOL]).float().to(DEVICE)             # (N_POOL,H) h at edit frame

# 8-dim (pos,vel) target: teleported positions + preserved original velocity (edits HDF5).
v_edits = h5py.File(edits.h5_path, "r")["velocities"][:, :, :N_OBJ, :].astype(np.float32)
tgt_pos = edits.positions[:N_POOL, EF, :N_OBJ, :].astype(np.float32)          # (N_POOL,2,2) teleported
tgt_vel = v_edits[:N_POOL, EF, :, :].astype(np.float32)                       # (N_POOL,2,2) preserved
TGT_POS = torch.from_numpy(tgt_pos.reshape(N_POOL, 4)).float().to(DEVICE)     # (N_POOL,4)
TGT8    = torch.from_numpy(np.concatenate([tgt_pos.reshape(N_POOL,4),
                                           tgt_vel.reshape(N_POOL,4)], 1)).float().to(DEVICE)  # (N_POOL,8)

# GT supervision: CLEAN post-edit observation sequence over K steps (step0 = decode at edit).
GT = torch.from_numpy(edits.clean_obs[:N_POOL, EF:EF+K, :]).float().to(DEVICE)   # (N_POOL,K,R)
edit_obj = edits.edit_object[:N_POOL]

# disjoint TRAIN / HELD-OUT indices
perm = np.random.RandomState(0).permutation(N_POOL)
TR_IDX = perm[:N_TRAIN]
HO_IDX = perm[N_TRAIN:N_TRAIN + N_HELD]
print(f"pool={N_POOL}  N_TRAIN={N_TRAIN}  N_HELD={N_HELD}  K={K}  disjoint={len(set(TR_IDX)&set(HO_IDX))==0}")

# Target normalization stats fit on TRAIN targets only (editor input conditioning).
_tm = TGT8[TR_IDX].mean(0, keepdim=True); _ts = TGT8[TR_IDX].std(0, keepdim=True) + 1e-6
def tnorm(t): return (t - _tm) / _ts

# ---- render TARGET & PRE-edit scenes for ghost / selectivity zones ----
sim = test.config["dataset"]["sim"]; OBS_RES = edits.obs_res
cfg1 = SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
                 n_objects=N_OBJ, radius=sim["radius"], n_frames=1, dt=sim["dt"], obs_res=sim["obs_res"],
                 refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
                 obs_noise_std=0.0, boundary="open", always_in_frustum=False)
REFL = np.array([sim["refl_min"], sim["refl_max"]], np.float32)
RAD  = np.array([sim["radius"]]*N_OBJ, np.float32)
COLc = np.tile(np.array([[1,1,1]], np.float32), (N_OBJ,1))
pre_pos = edits.positions[:N_POOL, EF-1, :N_OBJ, :].astype(np.float32)        # pre-edit positions

tgt_render_id  = np.zeros((N_POOL, OBS_RES), np.int64)
tgt_render_int = np.zeros((N_POOL, OBS_RES), np.float32)
pre_render_id  = np.zeros((N_POOL, OBS_RES), np.int64)
for i in range(N_POOL):
    sc = Scene(positions=tgt_pos[i][None], velocities=np.zeros((1,N_OBJ,2),np.float32),
               radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, rid, rint = render_scene(sc); tgt_render_id[i], tgt_render_int[i] = rid[0], rint[0]
    scp = Scene(positions=pre_pos[i][None], velocities=np.zeros((1,N_OBJ,2),np.float32),
                radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, ridp, _ = render_scene(scp); pre_render_id[i] = ridp[0]

# ghost zone: rays the edited object occupied pre-edit but should NOT after edit
ghost_mask = np.zeros((N_POOL, OBS_RES), bool)
# selectivity zone: rays owned by the NON-edited object in the target render (should be unchanged)
sel_mask   = np.zeros((N_POOL, OBS_RES), bool)
for i in range(N_POOL):
    oe = edit_obj[i]; on = 1 - oe
    ghost_mask[i] = (pre_render_id[i] == oe) & (tgt_render_id[i] != oe)
    sel_mask[i]   = (tgt_render_id[i] == on)
print(f"target/pre renders built. ghost rays total={int(ghost_mask.sum())}, sel rays total={int(sel_mask.sum())}")

---
## 3 — Shared machinery: batched rollout + obs-space metric suite

One batched differentiable rollout (`state_from_flat → decode → predict_step`, cudnn disabled so RNN backward works with frozen weights), and a single evaluator that maps any `h_edit` → full metric suite on a given index set: **obs-dist to GT post-edit rollout**, **ghost ratio**, **selectivity**, **persistence (per-step)**, **off-manifold residual**. All metrics computed here (no matplotlib); figures come later.

In [ ]:
# [3] Batched rollout (differentiable) + obs-space metric evaluator.
def rollout_batched(h_flat, k, model_=None):
    """h_flat:(B,H) -> obs:(B,k,R). Differentiable if h_flat requires grad. cudnn off for RNN backward."""
    m = model_ if model_ is not None else model
    with torch.backends.cudnn.flags(enabled=False):
        state = m.state_from_flat(h_flat)
        preds = [m.decode(state)]
        for _ in range(k - 1):
            p, state = m.predict_step(state); preds.append(p)
    return torch.stack(preds, 1)                                   # (B,k,R)

# Precompute per-sample masks/renders as tensors for vectorized metrics.
_GHOST = torch.from_numpy(ghost_mask).to(DEVICE)                   # (N_POOL,R) bool
_SEL   = torch.from_numpy(sel_mask).to(DEVICE)
_TGTINT = torch.from_numpy(tgt_render_int).float().to(DEVICE)      # (N_POOL,R) target render intensity

@torch.no_grad()
def eval_edit(h_edit, idx, k=K, model_=None):
    """Full obs-space metric suite for edited states h_edit (aligned to pool index `idx`)."""
    h_edit = h_edit.to(DEVICE)
    obs = rollout_batched(h_edit, k, model_=model_)               # (B,k,R)
    obs0 = obs[:, 0, :]                                            # direct-edit step
    gt   = GT[idx]                                                 # (B,k,R) clean post-edit GT
    ghost = _GHOST[idx]; sel = _SEL[idx]; tgtint = _TGTINT[idx]
    # unsteered reference rollout (frozen model, no edit) for ghost/selectivity normalization
    obs_u = rollout_batched(H0[idx], k, model_=model_)
    obs_u0 = obs_u[:, 0, :]
    # (i) obs distance to GT post-edit rollout (mean over K steps, and per-step curve)
    d_gt_step = ((obs - gt)**2).mean(dim=(0,2)).sqrt().cpu().numpy()          # (k,)
    d_gt      = float(((obs - gt)**2).mean().sqrt())
    # (ii) distance to TARGET render at direct-edit step
    d_tgt0    = float(((obs0 - tgtint)**2).mean().sqrt())
    # (iii) ghost ratio: intensity remaining in ghost zone / unsteered intensity there (lower=better)
    gnum = obs0[ghost].mean() if ghost.any() else torch.tensor(float('nan'), device=DEVICE)
    gden = obs_u0[ghost].mean().clamp_min(1e-6) if ghost.any() else torch.tensor(1.0, device=DEVICE)
    ghost_ratio = float(gnum / gden)
    ghost_step = np.array([float((obs[:,s,:][ghost].mean() / gden)) if ghost.any() else np.nan
                           for s in range(k)])
    # (iv) selectivity: RMS change in the NON-edited object's zone vs GT (lower=better; it should stay put)
    sel_err = float(((obs0[sel] - gt[:,0,:][sel])**2).mean().sqrt()) if sel.any() else np.nan
    sel_step = np.array([float(((obs[:,s,:][sel] - gt[:,s,:][sel])**2).mean().sqrt()) if sel.any() else np.nan
                         for s in range(k)])
    # (v) off-manifold residual of h_edit (global PCA)
    resid = float(offmanifold_residual(h_edit, subspace_dev).mean())
    return dict(obs=obs.cpu().numpy(), d_gt=d_gt, d_gt_step=d_gt_step, d_tgt0=d_tgt0,
                ghost_ratio=ghost_ratio, ghost_step=ghost_step, sel_err=sel_err, sel_step=sel_step,
                resid=resid)

# sanity: unsteered on held-out
_u = eval_edit(H0[HO_IDX], HO_IDX)
print(f"[unsteered HELD-OUT] d_gt={_u['d_gt']:.4f}  ghost={_u['ghost_ratio']:.3f}  "
      f"sel_err={_u['sel_err']:.4f}  resid={_u['resid']:.3f}  (real-state resid ref={real_res_global:.3f})")

---
## Variant A — Frozen-model learned editor  [information-presence test]

Train `E_θ:(h, tnorm(target8)) → Δh` on TRAIN edits; loss = obs-space MSE of the frozen `K`-step rollout from `h_edit=h+Δh` vs GT clean post-edit sequence, + `λ·off_manifold_residual(h_edit)`. **Model weights frozen — only `E_θ` trains.** We track TRAIN vs HELD-OUT obs loss every eval step (the memorization diagnostic). Then evaluate `E_θ` on held-out edits and stack head-to-head against every failed baseline.

In [ ]:
# [4] Train the frozen-model learned editor E_theta (obs-space rollout loss + manifold penalty).
class EditNet(nn.Module):
    def __init__(self, H, hidden=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(H + 8, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, H))
        # start near identity edit (small Delta h)
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)
    def forward(self, h, tgt8_norm):
        return self.net(torch.cat([h, tgt8_norm], 1))            # Delta h

def train_editor(tr_idx, ho_idx, model_, subspace_, n_iter=2500, bs=128, lr=5e-4,
                 lam_mani=0.01, log_every=250, seed=0):
    torch.manual_seed(seed)
    E = EditNet(H).to(DEVICE)
    opt = torch.optim.Adam(E.parameters(), lr=lr)
    hist = {"it": [], "tr": [], "ho": [], "resid": []}
    rng = np.random.RandomState(seed)
    for it in range(n_iter + 1):
        b = rng.choice(tr_idx, min(bs, len(tr_idx)), replace=False)
        dh = E(H0[b], tnorm(TGT8[b])); h_edit = H0[b] + dh
        roll = rollout_batched(h_edit, K, model_=model_)
        loss_obs  = ((roll - GT[b])**2).mean()
        loss_mani = offmanifold_residual(h_edit, subspace_).mean()
        loss = loss_obs + lam_mani * loss_mani
        opt.zero_grad(); loss.backward(); opt.step()
        if it % log_every == 0:
            with torch.no_grad():
                he_tr = H0[tr_idx] + E(H0[tr_idx], tnorm(TGT8[tr_idx]))
                l_tr = float(((rollout_batched(he_tr, K, model_=model_) - GT[tr_idx])**2).mean())
                he_ho = H0[ho_idx] + E(H0[ho_idx], tnorm(TGT8[ho_idx]))
                l_ho = float(((rollout_batched(he_ho, K, model_=model_) - GT[ho_idx])**2).mean())
                res  = float(offmanifold_residual(he_ho, subspace_).mean())
            hist["it"].append(it); hist["tr"].append(l_tr); hist["ho"].append(l_ho); hist["resid"].append(res)
            print(f"  it {it:5d} | train_obs {l_tr:.4f}  HELD-OUT obs {l_ho:.4f}  HO resid {res:.3f}")
    return E, hist

print(f"Training E_theta  (N_TRAIN={N_TRAIN}, frozen GRU, obs-space rollout loss)")
t0 = time.time()
editorA, histA = train_editor(TR_IDX, HO_IDX, model, subspace_dev, n_iter=2500)
print(f"done in {time.time()-t0:.1f}s")

# produce edited states for TRAIN and HELD-OUT
with torch.no_grad():
    hA_tr = H0[TR_IDX] + editorA(H0[TR_IDX], tnorm(TGT8[TR_IDX]))
    hA_ho = H0[HO_IDX] + editorA(H0[HO_IDX], tnorm(TGT8[HO_IDX]))

In [ ]:
# [5] Failed baselines on the SAME held-out edits: probe-pinv (pos), probe-pinv (pos,vel),
#     global-manifold (pos,vel), and obs-gradient (per-sample Adam on h vs GT clean rollout).
ho = HO_IDX
with torch.no_grad():
    # (a) position-only pseudo-inverse
    hB_posonly = inject_state(H0[ho], TGT_POS[ho], Ap, Ap_pinv, bp)
    # (b) joint (pos,vel) pseudo-inverse
    hB_posvel  = inject_state(H0[ho], TGT8[ho], Apv, Apv_pinv, bpv)
    # (c) global-manifold (pos,vel) via POCS against global PCA
    edit_fn_pv = lambda h, t: inject_state(h, t, Apv, Apv_pinv, bpv)
    hB_mani    = manifold_steer(H0[ho], TGT8[ho], edit_fn_pv, subspace_dev, n_iters=50)

# (d) obs-gradient editor: per-sample Adam on h minimizing K-step rollout vs GT clean obs (oracle, uses GT).
def obs_gradient_edit(h_init, gt_seq, n_iter=300, lr=0.05):
    h = h_init.clone().detach().requires_grad_(True)
    opt = torch.optim.Adam([h], lr=lr)
    for _ in range(n_iter):
        roll = rollout_batched(h, gt_seq.shape[1])
        loss = ((roll - gt_seq)**2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return h.detach()
t0 = time.time()
hB_obsgrad = obs_gradient_edit(H0[ho], GT[ho], n_iter=300, lr=0.05)
print(f"obs-gradient editor optimized {len(ho)} held-out samples in {time.time()-t0:.1f}s")

# Collect all held-out edited-state variants (learned editor + baselines + unsteered).
variants_ho = {
    "unsteered":            H0[ho],
    "probe-pinv (pos)":     hB_posonly,
    "probe-pinv (pos,vel)": hB_posvel,
    "manifold (pos,vel)":   hB_mani,
    "obs-gradient (oracle)":hB_obsgrad,
    "learned E_theta":      hA_ho,
}
metricsA = {nm: eval_edit(h, ho) for nm, h in variants_ho.items()}
print("held-out variants evaluated:", list(metricsA))

In [ ]:
# [6] Head-to-head HELD-OUT table + memorization diagnostic (train vs held-out for the learned editor).
print("="*92)
print(f"HELD-OUT OBS-SPACE HEAD-TO-HEAD  (N_HELD={len(ho)}, K={K})  — lower is better except where noted")
print("="*92)
print(f"{'variant':22s} {'d_gt(roll)':>10s} {'d_tgt(s0)':>10s} {'ghost':>7s} {'sel_err':>8s} {'resid':>7s}")
print("-"*92)
for nm, mt in metricsA.items():
    print(f"{nm:22s} {mt['d_gt']:10.4f} {mt['d_tgt0']:10.4f} {mt['ghost_ratio']:7.3f} "
          f"{mt['sel_err']:8.4f} {mt['resid']:7.3f}")
print("-"*92)
print(f"{'REAL-state resid ref':22s} {'':>10s} {'':>10s} {'':>7s} {'':>8s} {real_res_global:7.3f}")
print("ghost: fraction of pre-edit intensity remaining in vacated zone (1=unchanged, 0=object gone)")
print("sel_err: RMS error of the NON-edited object's rays vs GT (0=perfectly preserved)")
print("resid: off-manifold residual of h_edit (real states ~%.2f; obs-gradient historically ~15.7)" % real_res_global)

# Memorization diagnostic: learned editor on TRAIN vs HELD-OUT.
mA_tr = eval_edit(hA_tr, TR_IDX)
mA_ho = metricsA["learned E_theta"]
print("\n" + "="*60)
print("MEMORIZATION DIAGNOSTIC — learned E_theta: TRAIN vs HELD-OUT")
print("="*60)
print(f"{'':10s} {'d_gt':>8s} {'ghost':>7s} {'sel_err':>8s} {'resid':>7s}")
print(f"{'TRAIN':10s} {mA_tr['d_gt']:8.4f} {mA_tr['ghost_ratio']:7.3f} {mA_tr['sel_err']:8.4f} {mA_tr['resid']:7.3f}")
print(f"{'HELD-OUT':10s} {mA_ho['d_gt']:8.4f} {mA_ho['ghost_ratio']:7.3f} {mA_ho['sel_err']:8.4f} {mA_ho['resid']:7.3f}")
gap = mA_ho['d_gt'] - mA_tr['d_gt']
print(f"\ntrain->heldout d_gt gap = {gap:+.4f}   "
      f"(large gap => memorization; small gap => genuine controllability)")
print(f"unsteered held-out d_gt = {metricsA['unsteered']['d_gt']:.4f}  "
      f"=> held-out improvement over unsteered = {metricsA['unsteered']['d_gt']-mA_ho['d_gt']:+.4f}")

In [ ]:
# [7] Fig 1 — Variant A: (a) train vs held-out obs loss during training (memorization curve),
#     (b) held-out head-to-head bars (d_gt, ghost, sel_err).
plt.style.use("default")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
ax = axes[0]
# RMSE (sqrt of the stored MSE training loss) — matches d_gt units used in every table below.
ax.plot(histA["it"], np.sqrt(histA["tr"]), "-o", ms=4, color="#0072B2", label="TRAIN obs RMSE")
ax.plot(histA["it"], np.sqrt(histA["ho"]), "-s", ms=4, color="#D55E00", label="HELD-OUT obs RMSE")
ax.axhline(metricsA["unsteered"]["d_gt"], color="0.5", ls="--", lw=1, label="unsteered (held-out)")
ax.set_xlabel("iteration"); ax.set_ylabel("obs RMSE vs GT post-edit rollout")
ax.set_title("(a) memorization curve: train vs held-out"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
names = list(metricsA)
xx = np.arange(len(names)); w = 0.27
dgt = [metricsA[n]["d_gt"] for n in names]
gr  = [metricsA[n]["ghost_ratio"] for n in names]
se  = [metricsA[n]["sel_err"] for n in names]
ax.bar(xx-w, dgt, w, label="d_gt (rollout)", color="#0072B2")
ax.bar(xx,   gr,  w, label="ghost ratio",    color="#D55E00")
ax.bar(xx+w, se,  w, label="sel_err",        color="#009E73")
ax.set_xticks(xx); ax.set_xticklabels(names, rotation=30, ha="right", fontsize=7)
ax.set_title("(b) held-out head-to-head vs failed baselines"); ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
fig.suptitle("Fig 1 — Variant A: frozen learned editor — memorization curve + held-out head-to-head", y=1.02, fontsize=13)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_A_train_headtohead.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig); print("saved fig1_A_train_headtohead.png")

In [ ]:
# [8] Fig 2 — Variant A held-out step curves: (a) obs->GT rollout, (b) persistence/ghost, (c) selectivity.
steps = np.arange(K)
COL = {"unsteered":"0.5","probe-pinv (pos)":"#CC79A7","probe-pinv (pos,vel)":"#E69F00",
       "manifold (pos,vel)":"#56B4E9","obs-gradient (oracle)":"#D55E00","learned E_theta":"#0072B2"}
MK  = {"unsteered":None,"probe-pinv (pos)":"v","probe-pinv (pos,vel)":"^","manifold (pos,vel)":"D",
       "obs-gradient (oracle)":"x","learned E_theta":"o"}
def _lw(n): return 2.4 if n=="learned E_theta" else 1.3
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for n, mt in metricsA.items():
    axes[0].plot(steps, mt["d_gt_step"], color=COL[n], marker=MK[n], ms=3, lw=_lw(n), label=n)
    axes[1].plot(steps, mt["ghost_step"], color=COL[n], marker=MK[n], ms=3, lw=_lw(n), label=n)
    axes[2].plot(steps, mt["sel_step"], color=COL[n], marker=MK[n], ms=3, lw=_lw(n), label=n)
axes[0].set_title("(a) obs -> GT post-edit rollout (persistence)"); axes[0].set_ylabel("RMS(gen, GT)")
axes[1].set_title("(b) ghost ratio over rollout (revert test)"); axes[1].set_ylabel("ghost ratio")
axes[1].axhline(1.0, color="0.7", ls="--", lw=1); axes[1].axhline(0.0, color="0.7", ls=":", lw=1)
axes[2].set_title("(c) selectivity: non-edited obj error"); axes[2].set_ylabel("RMS(gen, GT) on other-obj rays")
for ax in axes: ax.set_xlabel("rollout step"); ax.grid(alpha=0.3); ax.legend(fontsize=6.5)
fig.suptitle("Fig 2 — Variant A held-out: persistence, ghost-revert, and selectivity over the rollout", y=1.02, fontsize=13)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_A_stepcurves.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig); print("saved fig2_A_stepcurves.png")

In [ ]:
# [9] Fig 3 — Variant A held-out observation-space evidence: 1D scans (a) + waterfalls (b).
# Pick HELD-OUT samples with big teleport AND a real ghost zone (the hard cases).
teleport = np.linalg.norm(tgt_pos.reshape(N_POOL,2,2) - pre_pos, axis=-1)[np.arange(N_POOL), edit_obj]
ho_arr = np.array(ho)
score = teleport[ho_arr] * (ghost_mask[ho_arr].sum(1) >= 3)
SAMP_LOCAL = np.argsort(score)[::-1][:3]              # positions within `ho`
SAMPLES = ho_arr[SAMP_LOCAL]                          # pool indices
print("held-out samples:", SAMPLES.tolist(), "teleport=", [round(float(teleport[s]),2) for s in SAMPLES])

order = ["unsteered","probe-pinv (pos,vel)","obs-gradient (oracle)","learned E_theta"]
rays = np.arange(OBS_RES)
def centroid(m):
    idx = np.where(m)[0]; return idx.mean() if idx.size else np.nan

# (a) 1D scans at direct-edit step (light theme is fine for scans; keep readable)
fig, axes = plt.subplots(len(SAMPLES), 1, figsize=(11, 3.0*len(SAMPLES)), squeeze=False)
for r, (loc, smp) in enumerate(zip(SAMP_LOCAL, SAMPLES)):
    ax = axes[r][0]
    ax.plot(rays, tgt_render_int[smp], color="k", ls="--", lw=1.6, label="TARGET render", zorder=6)
    ax.plot(rays, GT[smp, 0].cpu().numpy(), color="0.35", ls=":", lw=1.6, label="GT clean (step0)", zorder=5)
    gz = np.where(ghost_mask[smp])[0]
    if gz.size: ax.axvspan(gz.min()-.5, gz.max()+.5, color="red", alpha=0.10, label="ghost zone")
    sz = np.where(sel_mask[smp])[0]
    if sz.size: ax.axvspan(sz.min()-.5, sz.max()+.5, color="green", alpha=0.07, label="keep zone")
    for n in order:
        ax.plot(rays, metricsA[n]["obs"][loc, 0], color=COL[n], lw=2.2 if n=="learned E_theta" else 1.3,
                alpha=0.95 if n=="learned E_theta" else 0.8, label=n)
    ax.set_title(f"held-out sample {smp} (obj {edit_obj[smp]}, teleport={teleport[smp]:.2f})")
    ax.set_xlabel("ray"); ax.set_ylabel("intensity"); ax.set_ylim(-.02,1.05); ax.grid(alpha=.25); ax.legend(fontsize=7, ncol=3)
fig.suptitle("Fig 3a — Variant A held-out 1D scans at direct-edit step vs TARGET render", y=1.005, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3a_A_scans.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)

# (b) waterfalls (dark, master spec): N_CTX noisy context frames (clean for the GT column) above the dashed
#     edit-frame line; then the TEACHER-FORCED true edit-frame obs (clean_obs[EF], the SAME row in every
#     column = the edit target); then each column's free-run from EF+1 (the step-0 decode is EF, dropped so
#     every column maps to the same sim frames — fixes the +1). cmap=gray; green=target loc, red-dash=ghost loc.
from matplotlib.lines import Line2D
DARK_BG, DARK_TEXT, DARK_TICK, EDIT_LINE, TF_LINE = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850", "#FFD166"
N_CTX = min(6, EF)
wf_cols = ["GT (post-edit)"] + order
fig, axes = plt.subplots(len(SAMPLES), len(wf_cols), figsize=(2.9 * len(wf_cols), 3.3 * len(SAMPLES)), squeeze=False, facecolor=DARK_BG)
for r, (loc, smp) in enumerate(zip(SAMP_LOCAL, SAMPLES)):
    tgt_cx = centroid(tgt_render_id[smp] == edit_obj[smp]); pre_cx = centroid(pre_render_id[smp] == edit_obj[smp])
    tf_row = edits.clean_obs[smp, EF, :].astype(np.float32)                       # true post-edit obs AT EF (edit target)
    for c, name in enumerate(wf_cols):
        ax = axes[r][c]; ax.set_facecolor(DARK_BG)
        if name == "GT (post-edit)":
            ctx  = edits.clean_obs[smp, EF - N_CTX:EF, :].astype(np.float32)
            roll = edits.clean_obs[smp, EF + 1:EF + K, :].astype(np.float32)      # EF+1 .. EF+K-1 (free-run reference)
        else:
            ctx  = edits.obs[smp, EF - N_CTX:EF, :].astype(np.float32)            # noisy observed context
            roll = metricsA[name]["obs"][loc][1:]                                 # free-run EF+1 onward (drop step0=EF)
        panel = np.clip(np.concatenate([ctx, tf_row[None], roll], 0), 0, 1)       # ctx | EF (true) | free-run
        for sp in ax.spines.values(): sp.set_edgecolor(DARK_TICK)
        ax.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        ax.axhline(N_CTX - 0.5, color=EDIT_LINE, lw=1.2, ls="--", alpha=0.85)     # context -> edit frame
        ax.axhline(N_CTX + 0.5, color=TF_LINE, lw=1.1, ls=":", alpha=0.9)         # edit frame -> free-run (EF+1)
        if not np.isnan(tgt_cx): ax.axvline(tgt_cx, color="#00E676", lw=1.4, alpha=0.9)
        if not np.isnan(pre_cx): ax.axvline(pre_cx, color="#FF5252", ls="--", lw=1.4, alpha=0.9)
        if r == 0: ax.set_title(name, fontsize=8.5, color=("#00E676" if c == 0 else DARK_TEXT))
        if c == 0:
            ax.set_ylabel(f"smp {smp}\nsim frame", fontsize=8, color=DARK_TEXT)
            ax.set_yticks([0, N_CTX, N_CTX + 5, N_CTX + 10]); ax.set_yticklabels([EF - N_CTX, EF, EF + 5, EF + 10])
        else:
            ax.set_yticks([])
        ax.set_xlabel("ray", fontsize=8, color=DARK_TEXT); ax.tick_params(colors=DARK_TICK, labelsize=7)
handles = [Line2D([0], [0], color="#00E676", lw=2.2, label="object target (post-edit)"),
           Line2D([0], [0], color="#FF5252", ls="--", lw=2.2, label="pre-edit ghost location"),
           Line2D([0], [0], color=EDIT_LINE, ls="--", lw=2.2, label="edit frame"),
           Line2D([0], [0], color=TF_LINE, ls=":", lw=2.2, label="EF = true post-edit obs (edit target); rows below = free-run from the edited state")]
fig.legend(handles=handles, loc="upper center", ncol=2, fontsize=8.5, frameon=False, labelcolor=DARK_TEXT, bbox_to_anchor=(0.5, 0.995))
fig.suptitle("Fig 3b — Variant A held-out editor waterfalls (N_TRAIN={} editor); leftmost = GT (post-edit) reference".format(N_TRAIN),
             y=1.0, fontsize=11, color=DARK_TEXT)
fig.tight_layout(rect=[0, 0, 1, 0.94]); fig.savefig(f"{OUT}/fig3b_A_waterfalls.png", dpi=130, bbox_inches="tight", facecolor=DARK_BG)
display(fig); plt.close(fig); print("saved fig3a_A_scans.png, fig3b_A_waterfalls.png")

---
## 4 — Data-scaling: memorization vs controllability (the interpretation-guard separator)

The single sharpest test of "information present & reachable" vs "editor overfit the small set": sweep `N_TRAIN` and watch the **held-out** obs-error. If held-out tracks train (both low) at few-shot budget → controllability. If held-out stays flat near unsteered while train collapses → memorization at that budget; if held-out *improves with more data* → the information is present but not few-shot-reachable by this editor. All editors evaluated on the **same fixed held-out set**.

In [ ]:
# [10] Data-scaling sweep: train E_theta at several N_TRAIN, evaluate on the SAME fixed held-out set.
SCALE_NS = [64, 128, 256, 512, 1024, 2048]
# fixed held-out disjoint from ALL train pools: take the last N_HELD of the pool
HO_FIXED = perm[-N_HELD:]
avail = perm[:N_POOL - N_HELD]        # candidates for training (disjoint from HO_FIXED)
scale_rows = []
scale_metrics = {}
for n_tr in SCALE_NS:
    tr = avail[:n_tr]
    E, _h = train_editor(tr, HO_FIXED, model, subspace_dev, n_iter=2500, log_every=2600, seed=1)  # quiet
    with torch.no_grad():
        he_tr = H0[tr] + E(H0[tr], (TGT8[tr]-_tm)/_ts)
        he_ho = H0[HO_FIXED] + E(H0[HO_FIXED], (TGT8[HO_FIXED]-_tm)/_ts)
    m_tr = eval_edit(he_tr, tr); m_ho = eval_edit(he_ho, HO_FIXED)
    scale_metrics[n_tr] = m_ho
    scale_rows.append((n_tr, m_tr["d_gt"], m_ho["d_gt"], m_ho["ghost_ratio"], m_ho["sel_err"], m_ho["resid"]))
    print(f"N_TRAIN={n_tr:5d} | train d_gt {m_tr['d_gt']:.4f}  HELD-OUT d_gt {m_ho['d_gt']:.4f}  "
          f"ghost {m_ho['ghost_ratio']:.3f}  sel {m_ho['sel_err']:.4f}")

uns_ho_fixed = eval_edit(H0[HO_FIXED], HO_FIXED)
print("\n=== DATA-SCALING (held-out fixed) ===")
print(f"{'N_TRAIN':>8s} {'train d_gt':>11s} {'HO d_gt':>9s} {'HO ghost':>9s} {'HO sel':>8s} {'HO resid':>9s}")
for r in scale_rows:
    print(f"{r[0]:8d} {r[1]:11.4f} {r[2]:9.4f} {r[3]:9.3f} {r[4]:8.4f} {r[5]:9.3f}")
print(f"{'unsteered':>8s} {'—':>11s} {uns_ho_fixed['d_gt']:9.4f} {uns_ho_fixed['ghost_ratio']:9.3f} "
      f"{uns_ho_fixed['sel_err']:8.4f} {uns_ho_fixed['resid']:9.3f}")

In [ ]:
# [11] Fig 4 — Data-scaling: (a) train vs held-out d_gt vs N_TRAIN, (b) held-out ghost & sel vs N_TRAIN.
ns = [r[0] for r in scale_rows]
tr_d = [r[1] for r in scale_rows]; ho_d = [r[2] for r in scale_rows]
ho_g = [r[3] for r in scale_rows]; ho_s = [r[4] for r in scale_rows]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
ax = axes[0]
ax.plot(ns, tr_d, "-o", color="#0072B2", label="TRAIN d_gt")
ax.plot(ns, ho_d, "-s", color="#D55E00", label="HELD-OUT d_gt")
ax.axhline(uns_ho_fixed["d_gt"], color="0.5", ls="--", lw=1, label="unsteered (held-out)")
ax.set_xscale("log", base=2); ax.set_xticks(ns); ax.set_xticklabels(ns)
ax.set_xlabel("N_TRAIN (edit examples)"); ax.set_ylabel("obs RMS vs GT rollout")
ax.set_title("(a) memorization -> controllability as data grows"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax = axes[1]
ax.plot(ns, ho_g, "-o", color="#D55E00", label="held-out ghost ratio")
ax.plot(ns, ho_s, "-s", color="#009E73", label="held-out sel_err")
ax.axhline(uns_ho_fixed["ghost_ratio"], color="#D55E00", ls=":", lw=1)
ax.set_xscale("log", base=2); ax.set_xticks(ns); ax.set_xticklabels(ns)
ax.set_xlabel("N_TRAIN (edit examples)"); ax.set_ylabel("held-out metric")
ax.set_title("(b) held-out ghost & selectivity vs data"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.suptitle("Fig 4 — Variant A data-scaling: is control few-shot-reachable or data-hungry?", y=1.02, fontsize=13)
fig.tight_layout(); fig.savefig(f"{OUT}/fig4_A_datascaling.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig); print("saved fig4_A_datascaling.png")

---
## Variant B — Light fine-tune for editability  [inducibility test]

Now we let the **model** move (Variant A froze it). We ask: can a *light* fine-tune make a **FIXED, simple** editor work — i.e. is editability **inducible** from few examples?

**The two conditions being compared (so the reader isn't guessing):**
- **ORIG + fixed editor** — the *original* (un-fine-tuned) GRU, with the fixed `(pos,vel)` pseudo-inverse editor injecting the held-out target. This is the pre-existing "readable ≠ controllable" baseline.
- **FT + fixed editor** — the *fine-tuned* GRU with the **same** fixed editor. If the fine-tune induced editability, this beats ORIG on the obs-space suite.
- **FT unsteered** — fine-tuned model, no edit (sanity: fine-tuning must not by itself move the scene).

**What "fixed editor" means precisely.** The `(pos,vel)` linear probe used by the pseudo-inverse is **re-fit (detached / `no_grad`) on the *current* model's teacher-forced states** every `REFIT_EVERY=100` iters — it is *not* a frozen-weight probe. This keeps the editor honest: it always reads the model as it currently is, so any gain is the *model* becoming controllable, not the probe over-fitting a moving target through gradients. The injection is `h_edit = A⁺(target − b) + (h − A⁺A·h)` = replace the probe-readable `(pos,vel)` component with the target, keep the orthogonal remainder.

**Fine-tune objective.** For a batch of edit samples: differentiably teacher-force each to `edit_frame` with the *current* weights → `h_at_edit`; apply the fixed pseudo-inverse edit → `h_edit`; roll the dynamics forward **K=15 steps (multistep rollout)** and match the **GT clean post-edit observation sequence** (obs-space MSE). Plus a **next-step prediction-fidelity anchor** (`LAM_ANCHOR·` ordinary 1-step obs MSE on held-out test rollouts) so the fine-tune keeps the world model intact rather than collapsing it to satisfy the edit. **All GRU params train** (lr 2e-4) — this is a *light* fine-tune (short schedule, low lr, small budget), not a parameter-subset adapter.

**Eval.** Inject a **held-out** target (unseen edit) via that fixed pseudo-inverse and roll out K steps; score with the identical obs-space suite as Variant A (`d_gt`, `d_tgt(s0)`, `ghost`, `sel_err`, `resid` — see Definitions table). We also sweep the fine-tune **budget** (§ mirroring Variant A's `N_TRAIN` sweep) to test few-shot vs data-hungry.

**Then the payoff measurement (cell [14]):** re-run the fiber-collapse (`h ≈ g(pos,vel)`) + off-manifold geometry diagnostics on the fine-tuned model and compare to the ORIGINAL and to the known GRU references (fiber resid **0.337**, R²(h) **0.867** — `diagnostic-corrections`). If inducing editability made the state **more canonical** (fiber residual drops, embedding flattens), that directly supports the organizing hypothesis *editability ⟺ canonical state*.

> **Compute trade-off note.** Each Variant-B point is a full multistep-rollout fine-tune with an in-graph teacher-forced warm-up per step — substantially heavier than Variant A's frozen-editor training. The budget sweep therefore uses **~4 points** (vs A's 6) and a shorter schedule; it is enough to read the memorization-vs-inducibility trend, not a fine-grained scaling law.

In [ ]:
# [12] Variant B — light fine-tune of the GRU so a FIXED (pos,vel) pseudo-inverse editor induces the rollout.
#      Refactored into finetune_for_editability(...) so the budget sweep (cell [13c]) can reuse it.
import copy
edits_obs_t = torch.from_numpy(edits.obs[:N_POOL]).float().to(DEVICE)    # (N_POOL,T,R)

def warm_h_at_edit(m, idx):
    """Teacher-force obs[:EF] for samples idx -> flat hidden at edit frame (in-graph)."""
    with torch.backends.cudnn.flags(enabled=False):
        state = None
        obs = edits_obs_t[idx]                                            # (B,T,R)
        for t in range(EF):
            _, state = m.step(obs[:, t, :], state)
        return m.flat_state(state)                                        # (B,H)

def refit_pv_probe(m):
    """Re-fit a linear (pos,vel) probe on current model's teacher-forced states (DETACHED / no_grad).
    NOT a fixed-weight probe: it always reads the model as it currently is."""
    with torch.no_grad():
        obs_b = torch.from_numpy(test.obs[:2000]).float().to(DEVICE)
        x = torch.relu(m.encoder(obs_b[:, :-1, :])); hseq, _ = m.gru(x)
        Hs = hseq.reshape(-1, H)
        Y  = torch.from_numpy(posvel_tf[:2000].reshape(-1, 8)).float().to(DEVICE)
        vis = torch.from_numpy(vis_tf[:2000].reshape(-1)).to(DEVICE)
        Hs, Y = Hs[vis], Y[vis]
        Xa = torch.cat([Hs, torch.ones(Hs.shape[0],1,device=DEVICE)], 1)
        sol = torch.linalg.lstsq(Xa, Y).solution                          # (H+1,8)
        A = sol[:-1].T.contiguous(); b = sol[-1].contiguous()             # A:(8,H) b:(8,)
        A_pinv = torch.linalg.pinv(A)
    return A, b, A_pinv

def rollout_from_flat_m(m, h_flat, k):
    with torch.backends.cudnn.flags(enabled=False):
        state = m.state_from_flat(h_flat)
        preds = [m.decode(state)]
        for _ in range(k-1):
            p, state = m.predict_step(state); preds.append(p)
    return torch.stack(preds, 1)

test_obs_t = torch.from_numpy(test.obs[:4000]).float().to(DEVICE)

def finetune_for_editability(ft_train, n_iter=1500, bs=96, lr=2e-4, refit_every=100,
                             lam_anchor=1.0, log_every=250, verbose=True, seed=0):
    """Light all-param fine-tune. Objective = K-step multistep-rollout match to GT clean post-edit obs
    through a FIXED (re-fit, detached) pseudo-inverse editor, + next-step prediction-fidelity anchor.
    Returns (fine-tuned model, history). Model weights are copies; original `model` is untouched."""
    m = copy.deepcopy(model).to(DEVICE)
    for p_ in m.parameters(): p_.requires_grad_(True)
    m.train()
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    rng = np.random.RandomState(seed)
    hist = {"it": [], "edit": [], "anchor": [], "ho": []}
    A_pv, b_pv, A_pv_pinv = refit_pv_probe(m)
    for it in range(n_iter + 1):
        if it % refit_every == 0 and it > 0:
            A_pv, b_pv, A_pv_pinv = refit_pv_probe(m)
        b = rng.choice(ft_train, min(bs, len(ft_train)), replace=False)
        h_at = warm_h_at_edit(m, b)                                       # (B,H)
        h_perp = h_at - (h_at @ A_pv.T) @ A_pv_pinv.T
        h_inj = ((TGT8[b] - b_pv) @ A_pv_pinv.T) + h_perp                 # fixed pseudo-inverse edit
        roll = rollout_from_flat_m(m, h_inj, K)
        loss_edit = ((roll - GT[b])**2).mean()                            # K-step multistep obs MSE
        ab = rng.choice(test_obs_t.shape[0], min(bs, test_obs_t.shape[0]), replace=False)
        x = torch.relu(m.encoder(test_obs_t[ab][:, :-1, :]))
        with torch.backends.cudnn.flags(enabled=False):
            hseq, _ = m.gru(x)
        pred = m.decoder(hseq)
        loss_anchor = ((pred - test_obs_t[ab][:, 1:, :])**2).mean()       # next-step fidelity anchor
        loss = loss_edit + lam_anchor * loss_anchor
        opt.zero_grad(); loss.backward(); opt.step()
        if it % log_every == 0:
            with torch.no_grad():
                h_at_ho = warm_h_at_edit(m, HO_FIXED)
                hp = h_at_ho - (h_at_ho @ A_pv.T) @ A_pv_pinv.T
                h_inj_ho = ((TGT8[HO_FIXED] - b_pv) @ A_pv_pinv.T) + hp
                l_ho = float(((rollout_from_flat_m(m, h_inj_ho, K) - GT[HO_FIXED])**2).mean())
            hist["it"].append(it); hist["edit"].append(float(loss_edit.detach()))
            hist["anchor"].append(float(loss_anchor.detach())); hist["ho"].append(l_ho)
            if verbose:
                print(f"  it {it:4d} | edit {float(loss_edit):.4f}  anchor {float(loss_anchor):.5f}  HELD-OUT edit {l_ho:.4f}")
    m.eval()
    return m, hist

# ---- primary fine-tune at a documented budget (matched to the largest sweep point) ----
avail_ft = perm[:N_POOL - N_HELD]          # train pool (disjoint from HO_FIXED)
FT_BUDGET = N_TRAIN                         # primary Variant-B budget = Variant-A editor budget (few-shot)
                                           #   -> primary A/B waterfalls (Fig 3b, 5d) are at the SAME train size;
                                           #   larger budgets are explored in the sweeps (Fig 4, Fig 4B)
FT_TRAIN  = avail_ft[:FT_BUDGET]
print(f"Variant B primary fine-tune: budget={FT_BUDGET}, K={K} multistep, lr=2e-4, 1500 iters")
t0 = time.time()
model_ft, histB = finetune_for_editability(FT_TRAIN, n_iter=1500)
print(f"fine-tune done in {time.time()-t0:.1f}s")


In [ ]:
# [13] Evaluate fine-tuned model: fixed pseudo-inverse editor on held-out + prediction-fidelity check.
# Recompute teacher-forced states for the fine-tuned model (for probe/manifold/geometry).
@torch.no_grad()
def tf_states_ft(m, obs_np, n=4000):
    obs_b = torch.from_numpy(obs_np[:n]).float().to(DEVICE)
    x = torch.relu(m.encoder(obs_b[:, :-1, :]))
    with torch.backends.cudnn.flags(enabled=False):
        hseq, _ = m.gru(x)
    pred = m.decoder(hseq)
    return pred.cpu().numpy(), hseq.cpu().numpy()
pred_ft, states_ft = tf_states_ft(model_ft, test.obs)     # (n,39,H)

# prediction fidelity: next-step obs MSE, orig vs fine-tuned, on test.
with torch.no_grad():
    gt_next = test.obs[:4000, 1:, :]
    m_orig = torch.from_numpy(states_tf[:4000]).float()  # not needed; recompute orig preds directly
    obs_b = torch.from_numpy(test.obs[:4000]).float().to(DEVICE)
    x0 = torch.relu(model.encoder(obs_b[:, :-1, :]))
    with torch.backends.cudnn.flags(enabled=False):
        h0seq, _ = model.gru(x0)
    pred_orig = model.decoder(h0seq).cpu().numpy()
fid_orig = float(np.sqrt(((pred_orig - gt_next)**2).mean()))
fid_ft   = float(np.sqrt(((pred_ft[:4000] - gt_next)**2).mean()))

# fixed pseudo-inverse editor on held-out, for BOTH models (re-fit probe per model, no editability training on orig).
def fixed_editor_eval(m, states_bank):
    # refit probe on this model's states
    vis = vis_tf[:states_bank.shape[0]].reshape(-1)
    Hs = states_bank.reshape(-1, H)[vis]; Y = posvel_tf[:states_bank.shape[0]].reshape(-1,8)[vis]
    Xa = np.concatenate([Hs, np.ones((Hs.shape[0],1),np.float32)],1)
    sol = np.linalg.lstsq(Xa, Y, rcond=None)[0]
    A = torch.from_numpy(sol[:-1].T.copy()).float().to(DEVICE); b = torch.from_numpy(sol[-1].copy()).float().to(DEVICE)
    A_pinv = torch.linalg.pinv(A)
    h_at = warm_h_at_edit(m, HO_FIXED).detach()
    hp = h_at - (h_at @ A.T) @ A_pinv.T
    h_inj = ((TGT8[HO_FIXED] - b) @ A_pinv.T) + hp
    return eval_edit(h_inj, HO_FIXED, model_=m), (A, b, A_pinv)

mB_orig, _ = fixed_editor_eval(model, states_tf[:4000])
mB_ft, ft_probe = fixed_editor_eval(model_ft, states_ft)
uns_ft = eval_edit(warm_h_at_edit(model_ft, HO_FIXED).detach(), HO_FIXED, model_=model_ft)

print("=== VARIANT B: prediction fidelity (did the fine-tune keep the world model?) ===")
print(f"  next-step obs RMSE  orig={fid_orig:.4f}   fine-tuned={fid_ft:.4f}   (delta {fid_ft-fid_orig:+.4f})")
print("\n=== VARIANT B: FIXED pseudo-inverse editor on HELD-OUT (does the fine-tune INDUCE editability?) ===")
print(f"{'model':16s} {'d_gt':>8s} {'d_tgt(s0)':>10s} {'ghost':>7s} {'sel_err':>8s} {'resid':>7s}")
print(f"{'ORIG + fixed':16s} {mB_orig['d_gt']:8.4f} {mB_orig['d_tgt0']:10.4f} {mB_orig['ghost_ratio']:7.3f} {mB_orig['sel_err']:8.4f} {mB_orig['resid']:7.3f}")
print(f"{'FT + fixed':16s} {mB_ft['d_gt']:8.4f} {mB_ft['d_tgt0']:10.4f} {mB_ft['ghost_ratio']:7.3f} {mB_ft['sel_err']:8.4f} {mB_ft['resid']:7.3f}")
print(f"{'FT unsteered':16s} {uns_ft['d_gt']:8.4f} {uns_ft['d_tgt0']:10.4f} {uns_ft['ghost_ratio']:7.3f} {uns_ft['sel_err']:8.4f} {uns_ft['resid']:7.3f}")
print(f"\nheld-out improvement from fine-tune (fixed editor d_gt): {mB_orig['d_gt']-mB_ft['d_gt']:+.4f}")

---
### 4B — Variant B fine-tune budget sweep (mirror of Variant A's N_TRAIN sweep)

Same question as Fig 4, now for the *inducibility* route: does the FT+fixed-editor held-out quality improve as we fine-tune on more edit examples, or is it flat (memorization)? **~4 budget points** (fine-tuning is much heavier than editor-training — see the compute note above). All points evaluated on the SAME fixed held-out set with the identical obs-space suite (`d_gt`, `ghost`, `sel_err`, `resid`), directly comparable to Fig 4.

In [ ]:
# [13c] Variant-B fine-tune budget sweep: fine-tune at several budgets, eval FT+fixed editor on fixed held-out.
#       Mirrors Variant A's N_TRAIN sweep (cell [10]); ~4 points because fine-tuning is heavy (compute note).
FT_SWEEP_NS = [128, 256, 512, 1024]
ftB_rows = []; ftB_metrics = {}
for n_ft in FT_SWEEP_NS:
    tr = avail_ft[:n_ft]
    t0 = time.time()
    m_s, _h = finetune_for_editability(tr, n_iter=1500, verbose=False, seed=1)
    # refit probe on THIS FT model's states, then FT + fixed-editor held-out eval (identical to cell [13])
    states_s = tf_states_ft(m_s, test.obs)[1]
    m_eval, _ = fixed_editor_eval(m_s, states_s)
    ftB_metrics[n_ft] = m_eval
    ftB_rows.append((n_ft, m_eval["d_gt"], m_eval["d_tgt0"], m_eval["ghost_ratio"], m_eval["sel_err"], m_eval["resid"]))
    del m_s
    torch.cuda.empty_cache()
    print(f"FT_BUDGET={n_ft:5d} | d_gt {m_eval['d_gt']:.4f}  ghost {m_eval['ghost_ratio']:.3f}  "
          f"sel {m_eval['sel_err']:.4f}  resid {m_eval['resid']:.3f}  ({time.time()-t0:.0f}s)")

print("\n" + "="*84)
print(f"VARIANT B — FINE-TUNE BUDGET SWEEP (FT + fixed editor, held-out N={len(HO_FIXED)}, K={K})")
print("="*84)
print(f"{'FT_BUDGET':>9s} {'d_gt':>8s} {'d_tgt(s0)':>10s} {'ghost':>7s} {'sel_err':>8s} {'resid':>7s}")
for r in ftB_rows:
    print(f"{r[0]:9d} {r[1]:8.4f} {r[2]:10.4f} {r[3]:7.3f} {r[4]:8.4f} {r[5]:7.3f}")
# reference rows: ORIG+fixed editor (no fine-tune) and FT-unsteered from cell [13]
print(f"{'ORIG(ref)':>9s} {mB_orig['d_gt']:8.4f} {mB_orig['d_tgt0']:10.4f} {mB_orig['ghost_ratio']:7.3f} "
      f"{mB_orig['sel_err']:8.4f} {mB_orig['resid']:7.3f}   <- original GRU, fixed editor (no fine-tune)")


In [ ]:
# [13d] Fig 4B — Variant B budget sweep vs the ORIG (no-fine-tune) baseline. Same axes/units as Fig 4.
ns_b = [r[0] for r in ftB_rows]
d_b  = [r[1] for r in ftB_rows]; g_b = [r[3] for r in ftB_rows]; s_b = [r[4] for r in ftB_rows]
plt.style.use("default")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
ax = axes[0]
ax.plot(ns_b, d_b, "-s", color="#0072B2", label="FT + fixed editor: HELD-OUT d_gt")
ax.axhline(mB_orig["d_gt"], color="#999999", ls="--", lw=1.2, label="ORIG + fixed editor (no FT)")
ax.axhline(uns_ft["d_gt"], color="0.5", ls=":", lw=1, label="FT unsteered")
ax.set_xscale("log", base=2); ax.set_xticks(ns_b); ax.set_xticklabels(ns_b)
ax.set_xlabel("fine-tune budget (edit examples)"); ax.set_ylabel("held-out obs RMSE vs GT rollout")
ax.set_title("(a) does more fine-tune data induce editability?"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax = axes[1]
ax.plot(ns_b, g_b, "-o", color="#D55E00", label="FT held-out ghost ratio")
ax.plot(ns_b, s_b, "-s", color="#009E73", label="FT held-out sel_err")
ax.axhline(mB_orig["ghost_ratio"], color="#D55E00", ls="--", lw=1)
ax.axhline(mB_orig["sel_err"], color="#009E73", ls="--", lw=1)
ax.set_xscale("log", base=2); ax.set_xticks(ns_b); ax.set_xticklabels(ns_b)
ax.set_xlabel("fine-tune budget (edit examples)"); ax.set_ylabel("held-out metric (dashed = ORIG ref)")
ax.set_title("(b) held-out ghost & selectivity vs fine-tune budget"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.suptitle("Fig 4B — Variant B fine-tune budget sweep (mirror of Fig 4): inducible or memorizing?", y=1.02, fontsize=13)
fig.tight_layout(); fig.savefig(f"{OUT}/fig4B_B_datascaling.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig); print("saved fig4B_B_datascaling.png")


In [ ]:
# [14] Canonicity re-measurement: fiber-collapse (h ≈ g(pos,vel)) + linear (pos,vel) recoverability,
#      ORIGINAL vs FINE-TUNED states. Did inducing editability make the state more canonical?
def fit_g_residual(states_bank, kind="mlp", hidden=512, n_epochs=120, lr=1.5e-3):
    """Fit g:(pos,vel)->h; return residual fraction ||h-g||/||h|| and R2 on h."""
    n = states_bank.shape[0]
    vis = vis_tf[:n].reshape(-1)
    Y = torch.from_numpy(states_bank.reshape(-1, H)[vis].astype(np.float32)).to(DEVICE)
    X = torch.from_numpy(posvel_tf[:n].reshape(-1, 8)[vis].astype(np.float32)).to(DEVICE)
    hn2 = (Y**2).sum()
    if kind == "linear":
        Xa = torch.cat([X, torch.ones(X.shape[0],1,device=DEVICE)],1)
        sol = torch.linalg.lstsq(Xa, Y).solution; pred = Xa @ sol
    else:
        net = nn.Sequential(nn.Linear(8,hidden),nn.ReLU(),nn.Linear(hidden,hidden),nn.ReLU(),nn.Linear(hidden,H)).to(DEVICE)
        o = torch.optim.Adam(net.parameters(), lr=lr); bs=4096
        for ep in range(n_epochs):
            pm = torch.randperm(X.shape[0], device=DEVICE)
            for i in range(0, X.shape[0], bs):
                idx = pm[i:i+bs]; l = ((net(X[idx])-Y[idx])**2).mean()
                o.zero_grad(); l.backward(); o.step()
        with torch.no_grad(): pred = net(X)
    resid2 = ((pred-Y)**2).sum()
    rf = float((resid2/hn2).sqrt()); mu = Y.mean(0,keepdim=True)
    r2 = float(1 - resid2/((Y-mu)**2).sum())
    return rf, r2

def lin_pv_r2(states_bank):
    """Linear (pos,vel) probe R2 (readability) on a state bank."""
    n = states_bank.shape[0]; vis = vis_tf[:n].reshape(-1)
    Hs = states_bank.reshape(-1,H)[vis]; Y = posvel_tf[:n].reshape(-1,8)[vis]
    Xa = np.concatenate([Hs, np.ones((Hs.shape[0],1),np.float32)],1)
    sol = np.linalg.lstsq(Xa, Y, rcond=None)[0]; pred = Xa @ sol
    mu = Y.mean(0); ss_res=((pred-Y)**2).sum(0); ss_tot=((Y-mu)**2).sum(0)
    r2 = 1 - ss_res/np.maximum(ss_tot,1e-12)
    return float(r2[:4].mean()), float(r2[4:].mean())

# off-manifold reference for each model's own state bank
def own_resid(states_bank):
    ss = fit_state_subspace(states_bank, var_threshold=SUBSPACE_VAR)
    ss = replace(ss, mean=ss.mean.to(DEVICE), basis=ss.basis.to(DEVICE),
                 explained_variance_ratio=ss.explained_variance_ratio.to(DEVICE))
    r = float(offmanifold_residual(torch.from_numpy(states_bank.reshape(-1,H)[:8000]).float().to(DEVICE), ss).mean())
    return r, ss.n_components

geo = {}
for tag, bank in [("ORIGINAL", states_tf[:4000]), ("FINE-TUNED", states_ft)]:
    rf_lin, r2_lin = fit_g_residual(bank, "linear")
    rf_mlp, r2_mlp = fit_g_residual(bank, "mlp")
    pos_r2, vel_r2 = lin_pv_r2(bank)
    res_own, ncomp = own_resid(bank)
    geo[tag] = dict(rf_lin=rf_lin, r2_lin=r2_lin, rf_mlp=rf_mlp, r2_mlp=r2_mlp,
                    pos_r2=pos_r2, vel_r2=vel_r2, res_own=res_own, ncomp=ncomp)

print("=== CANONICITY RE-MEASUREMENT: ORIGINAL vs FINE-TUNED ===")
print("(ref: GRU fiber resid 0.337 / R2(h) 0.867 (MLP) from research/scratch/2026-07-08-diagnostic-corrections.md Sec.2;")
print(" ORIGINAL below should reproduce that order-of-magnitude on this state bank)")
print(f"{'metric':34s} {'ORIGINAL':>10s} {'FINE-TUNED':>11s}")
rows = [("fiber resid ||h-g||/||h|| (linear)","rf_lin"),
        ("fiber resid ||h-g||/||h|| (MLP)","rf_mlp"),
        ("R2(h) from (pos,vel)  (MLP)","r2_mlp"),
        ("linear pos R2 (readability)","pos_r2"),
        ("linear vel R2 (readability)","vel_r2"),
        ("off-manifold resid (own PCA)","res_own"),
        ("PCA comps @90% var","ncomp")]
for lbl, k in rows:
    print(f"{lbl:34s} {geo['ORIGINAL'][k]:10.4f} {geo['FINE-TUNED'][k]:11.4f}")
print("\nInterpretation: LOWER MLP fiber resid / HIGHER R2(h) / LOWER off-manifold resid / FEWER comps")
print("=> state became MORE canonical (editability ⟺ canonical state). Otherwise editability was induced")
print("   WITHOUT canonicalizing the state (decouples the hypothesis).")

In [ ]:
# [15] Fig 5 — Variant B: (a) fine-tune curves (RMSE), (b) held-out fixed-editor orig-vs-FT bars, (c) canonicity bars.
plt.style.use("default")
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
ax = axes[0]
# RMSE (sqrt of stored MSE) — same units as d_gt everywhere else.
ax.plot(histB["it"], np.sqrt(histB["edit"]), "-o", ms=4, color="#0072B2", label="edit RMSE (train)")
ax.plot(histB["it"], np.sqrt(histB["ho"]),   "-s", ms=4, color="#D55E00", label="edit RMSE (held-out)")
ax.plot(histB["it"], np.sqrt(histB["anchor"]),"-^", ms=4, color="#009E73", label="prediction anchor RMSE")
ax.set_xlabel("fine-tune iteration"); ax.set_ylabel("obs RMSE")
ax.set_title("(a) fine-tune: editability + fidelity anchor"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
labels = ["d_gt","ghost","sel_err"]; xx = np.arange(3); w=0.35
orig_v = [mB_orig["d_gt"], mB_orig["ghost_ratio"], mB_orig["sel_err"]]
ft_v   = [mB_ft["d_gt"],   mB_ft["ghost_ratio"],   mB_ft["sel_err"]]
ax.bar(xx-w/2, orig_v, w, label="ORIG + fixed editor", color="#999999")
ax.bar(xx+w/2, ft_v,   w, label="FT + fixed editor", color="#0072B2")
ax.set_xticks(xx); ax.set_xticklabels(labels); ax.set_ylabel("held-out metric (RMSE / ratio)")
ax.set_title("(b) held-out fixed-editor quality"); ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")

ax = axes[2]
ckeys = [("MLP fiber\nresid","rf_mlp"),("off-mani\nresid","res_own"),("lin vel\nR2","vel_r2")]
xx = np.arange(len(ckeys)); w=0.35
ax.bar(xx-w/2, [geo["ORIGINAL"][k] for _,k in ckeys], w, label="ORIGINAL", color="#999999")
ax.bar(xx+w/2, [geo["FINE-TUNED"][k] for _,k in ckeys], w, label="FINE-TUNED", color="#D55E00")
ax.set_xticks(xx); ax.set_xticklabels([l for l,_ in ckeys]); ax.set_title("(c) canonicity re-measurement")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
fig.suptitle("Fig 5 — Variant B: light fine-tune for editability + canonicity re-measurement", y=1.02, fontsize=13)
fig.tight_layout(); fig.savefig(f"{OUT}/fig5_B_finetune.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig); print("saved fig5_B_finetune.png")

# (5d) waterfalls with a GT (post-edit) REFERENCE column (leftmost). Same SAMPLES as Fig 3b so A and B
#      show the SAME held-out cases; both primary waterfalls at their documented budgets
#      (A editor N_TRAIN, B fine-tune FT_BUDGET). Dark theme (simulator output).
A_ft, b_ft, Ap_ft = ft_probe
with torch.no_grad():
    h_at_s = warm_h_at_edit(model_ft, SAMPLES).detach()
    hp_s = h_at_s - (h_at_s @ A_ft.T) @ Ap_ft.T
    h_inj_s = ((TGT8[SAMPLES] - b_ft) @ Ap_ft.T) + hp_s
    ft_obs = rollout_from_flat_m(model_ft, h_inj_s, K).cpu().numpy()
wf_cols = ["GT (post-edit)", "ORIG unsteered", "ORIG + fixed edit", "FT + fixed edit"]
# master-spec waterfalls: N_CTX noisy context (clean for GT) | dashed edit line | TEACHER-FORCED true EF obs
# (shared) | dotted free-run line | each column's free-run from EF+1 (drop step0=EF). cmap=gray.
from matplotlib.lines import Line2D
DARK_BG, DARK_TEXT, DARK_TICK, EDIT_LINE, TF_LINE = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850", "#FFD166"
N_CTX = min(6, EF)
fig, axes = plt.subplots(len(SAMPLES), len(wf_cols), figsize=(2.9 * len(wf_cols), 3.3 * len(SAMPLES)), squeeze=False, facecolor=DARK_BG)
for r, smp in enumerate(SAMPLES):
    loc = int(np.where(ho_arr == smp)[0][0])
    tgt_cx = centroid(tgt_render_id[smp] == edit_obj[smp]); pre_cx = centroid(pre_render_id[smp] == edit_obj[smp])
    tf_row = edits.clean_obs[smp, EF, :].astype(np.float32)                       # true post-edit obs AT EF (edit target)
    model_rolls = [metricsA["unsteered"]["obs"][loc], metricsA["probe-pinv (pos,vel)"]["obs"][loc], ft_obs[r]]
    for c, name in enumerate(wf_cols):
        ax = axes[r][c]; ax.set_facecolor(DARK_BG)
        if name == "GT (post-edit)":
            ctx  = edits.clean_obs[smp, EF - N_CTX:EF, :].astype(np.float32)
            roll = edits.clean_obs[smp, EF + 1:EF + K, :].astype(np.float32)      # EF+1 .. EF+K-1 (free-run reference)
        else:
            ctx  = edits.obs[smp, EF - N_CTX:EF, :].astype(np.float32)            # noisy observed context
            roll = model_rolls[c - 1][1:]                                         # free-run EF+1 onward (drop step0=EF)
        panel = np.clip(np.concatenate([ctx, tf_row[None], roll], 0), 0, 1)       # ctx | EF (true) | free-run
        for sp in ax.spines.values(): sp.set_edgecolor(DARK_TICK)
        ax.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        ax.axhline(N_CTX - 0.5, color=EDIT_LINE, lw=1.2, ls="--", alpha=0.85)     # context -> edit frame
        ax.axhline(N_CTX + 0.5, color=TF_LINE, lw=1.1, ls=":", alpha=0.9)         # edit frame -> free-run (EF+1)
        if not np.isnan(tgt_cx): ax.axvline(tgt_cx, color="#00E676", lw=1.4, alpha=0.9)
        if not np.isnan(pre_cx): ax.axvline(pre_cx, color="#FF5252", ls="--", lw=1.4, alpha=0.9)
        if r == 0: ax.set_title(name, fontsize=8.5, color=("#00E676" if c == 0 else DARK_TEXT))
        if c == 0:
            ax.set_ylabel(f"smp {smp}\nsim frame", fontsize=8, color=DARK_TEXT)
            ax.set_yticks([0, N_CTX, N_CTX + 5, N_CTX + 10]); ax.set_yticklabels([EF - N_CTX, EF, EF + 5, EF + 10])
        else:
            ax.set_yticks([])
        ax.set_xlabel("ray", fontsize=8, color=DARK_TEXT); ax.tick_params(colors=DARK_TICK, labelsize=7)
handles = [Line2D([0], [0], color="#00E676", lw=2.2, label="object target (post-edit)"),
           Line2D([0], [0], color="#FF5252", ls="--", lw=2.2, label="pre-edit ghost location"),
           Line2D([0], [0], color=EDIT_LINE, ls="--", lw=2.2, label="edit frame"),
           Line2D([0], [0], color=TF_LINE, ls=":", lw=2.2, label="EF = true post-edit obs (edit target); rows below = free-run from the edited state")]
fig.legend(handles=handles, loc="upper center", ncol=2, fontsize=8.5, frameon=False, labelcolor=DARK_TEXT, bbox_to_anchor=(0.5, 0.995))
fig.suptitle("Fig 5d — Variant B held-out waterfalls (FT_BUDGET={}); leftmost = GT (post-edit) reference; "
             "does fine-tune make the fixed editor work?".format(FT_BUDGET), y=1.0, fontsize=10.5, color=DARK_TEXT)
fig.tight_layout(rect=[0, 0, 1, 0.94]); fig.savefig(f"{OUT}/fig5d_B_waterfalls.png", dpi=130, bbox_inches="tight", facecolor=DARK_BG)
display(fig); plt.close(fig); print("saved fig5d_B_waterfalls.png")

---
## Summary

See `research/scratch/2026-07-09-learn-to-edit.md` for the verdict. All obs-space errors are **RMSE** (Definitions table); every editing comparison figure carries a **GT (post-edit) reference column**; the primary A and B waterfalls are at the **same train size** (`N_TRAIN = FT_BUDGET`). Structure:

- **Variant A (frozen learned editor):** cell [6] head-to-head + memorization diagnostic; Fig 1–3 (held-out quality, persistence/ghost/selectivity, waterfalls with GT column); Fig 4 data-scaling. *Does an amortized `E_θ` induce clean, persistent, **selective** edits on held-out edits, or memorize?*
- **Variant B (light fine-tune):** cell [13] fixed-editor held-out + prediction fidelity (same RMSE suite as A); **cell [13c]/Fig 4B fine-tune budget sweep** (mirror of Fig 4); cell [14] canonicity re-measurement (fiber collapse + geometry, orig vs fine-tuned, vs the `diagnostic-corrections` references); Fig 5 (RMSE curves + GT-column waterfalls). *Is editability **inducible** via a light fine-tune, and if so did the state become **more canonical**?*

Interpretation guard applied throughout: the **train↔held-out gap**, **data/budget-scaling curves** (Fig 4 & 4B), and **selectivity** separate genuine controllability from memorization; the **off-manifold residual** of `h_edit` separates an on-manifold edit from the obs-gradient oracle's off-manifold shortcut.